# 05 — KPI Computation & Final Tableau Load Preparation

**Project:** Online Retail Revenue Intelligence  
**Team:** Hopper_DataDrift_OnlineRetailAnalytics | Newton School of Technology  

**Objective:**  
Compute all KPIs from the cleaned dataset and export final, Tableau-ready CSV files. Every KPI is defined explicitly so it can be reproduced and explained during viva.

**KPIs Computed:**
1. Total Revenue
2. Average Order Value (AOV)
3. Cancellation Rate
4. Monthly Revenue + MoM Growth %
5. Revenue by Country
6. Top Products by Revenue
7. Customer Lifetime Value (proxy)
8. RFM Segment Distribution

**Exports:**
- `data/processed/kpi_summary.csv` — single-row KPI snapshot
- `data/processed/monthly_kpis.csv` — monthly aggregates for trend charts
- `data/processed/country_kpis.csv` — country aggregates for map view
- `data/processed/product_kpis.csv` — product-level aggregates
- `data/processed/rfm_segments.csv` — customer RFM data
- `data/processed/tableau_main.csv` — full transaction-level file for Tableau (all features)

## 5.1 — Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT   = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR  = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df       = pd.read_csv(PROCESSED_DIR / 'cleaned_retail.csv', parse_dates=['invoicedate'])
df_sales = df[~df['is_cancelled']].copy()

print(f'Full dataset : {df.shape}')
print(f'Sales only   : {df_sales.shape}')

Full dataset : (1027017, 18)
Sales only   : (1007913, 18)


## 5.2 — KPI 1 to 3: Core Business KPIs

In [2]:
# --- KPI 1: Total Revenue ---
# Definition: Sum of (Quantity × Price) for all non-cancelled transactions
total_revenue = df_sales['revenue'].sum()

# --- KPI 2: Average Order Value (AOV) ---
# Definition: Total Revenue / Number of unique non-cancelled invoices
unique_orders = df_sales['invoice'].nunique()
aov = total_revenue / unique_orders

# --- KPI 3: Cancellation Rate ---
# Definition: Cancelled invoices / Total invoices (including cancelled)
total_invoices     = df['invoice'].nunique()
cancelled_invoices = df[df['is_cancelled']]['invoice'].nunique()
cancellation_rate  = cancelled_invoices / total_invoices * 100

# Revenue lost to cancellations
revenue_lost = df[df['is_cancelled']]['abs_revenue'].sum()

# --- KPI 4: Unique Customers (known) ---
unique_customers = df_sales[df_sales['customer_id'] != 'GUEST']['customer_id'].nunique()

# --- KPI 5: Unique Products ---
unique_products = df_sales['description'].nunique()

print('=== CORE KPI DASHBOARD ===')
print(f'  Total Revenue         : £{total_revenue:>15,.2f}')
print(f'  Average Order Value   : £{aov:>15,.2f}')
print(f'  Cancellation Rate     :  {cancellation_rate:>14.1f}%')
print(f'  Revenue Lost (cancels): £{revenue_lost:>15,.2f}')
print(f'  Unique Customers      :  {unique_customers:>14,}')
print(f'  Unique Products       :  {unique_products:>14,}')
print(f'  Total Orders          :  {unique_orders:>14,}')

=== CORE KPI DASHBOARD ===
  Total Revenue         : £  20,476,260.45
  Average Order Value   : £         510.92
  Cancellation Rate     :            17.1%
  Revenue Lost (cancels): £   1,462,797.75
  Unique Customers      :           5,878
  Unique Products       :           5,356
  Total Orders          :          40,077


## 5.3 — Export: KPI Summary

In [3]:
kpi_summary = pd.DataFrame([{
    'total_revenue'      : round(total_revenue, 2),
    'average_order_value': round(aov, 2),
    'cancellation_rate_pct': round(cancellation_rate, 2),
    'revenue_lost_to_cancellations': round(revenue_lost, 2),
    'unique_customers'   : unique_customers,
    'unique_products'    : unique_products,
    'total_orders'       : unique_orders
}])

kpi_summary.to_csv(PROCESSED_DIR / 'kpi_summary.csv', index=False)
print('Saved: kpi_summary.csv')
kpi_summary

Saved: kpi_summary.csv


,total_revenue,average_order_value,cancellation_rate_pct,revenue_lost_to_cancellations,unique_customers,unique_products,total_orders
0,20476260.45,510.92,17.14,1462797.75,5878,5356,40077


## 5.4 — Export: Monthly KPIs

In [4]:
monthly = (
    df_sales.groupby(['year','month','month_name']).agg(
        revenue       = ('revenue',  'sum'),
        orders        = ('invoice',  'nunique'),
        items_sold    = ('quantity', 'sum'),
        unique_customers = ('customer_id', 'nunique')
    ).reset_index().sort_values(['year','month'])
)
monthly['aov']        = (monthly['revenue'] / monthly['orders']).round(2)
monthly['mom_growth'] = monthly['revenue'].pct_change().mul(100).round(2)
monthly['period']     = monthly['year'].astype(str) + '-' + monthly['month'].astype(str).str.zfill(2)

monthly.to_csv(PROCESSED_DIR / 'monthly_kpis.csv', index=False)
print('Saved: monthly_kpis.csv')
monthly[['period','revenue','orders','aov','mom_growth']].to_string(index=False)

Saved: monthly_kpis.csv


' period     revenue  orders    aov  mom_growth\n2009-12  822483.950    1682 488.99         NaN\n2010-01  651155.112    1105 589.28      -20.83\n2010-02  551504.726    1201 459.20      -15.30\n2010-03  830915.261    1681 494.30       50.66\n2010-04  678875.252    1462 464.35      -18.30\n2010-05  657705.500    1500 438.47       -3.12\n2010-06  749537.310    1645 455.65       13.96\n2010-07  648810.270    1529 424.34      -13.44\n2010-08  695251.910    1425 487.90        7.16\n2010-09  921696.991    1839 501.19       32.57\n2010-10 1161902.220    2301 504.96       26.06\n2010-11 1464293.142    2747 533.05       26.03\n2010-12  821452.730    1559 526.91      -43.90\n2011-01  689811.610    1086 635.19      -16.03\n2011-02  522545.560    1100 475.04      -24.25\n2011-03  716215.260    1454 492.58       37.06\n2011-04  536968.491    1246 430.95      -25.03\n2011-05  769296.610    1681 457.64       43.27\n2011-06  760547.010    1533 496.12       -1.14\n2011-07  718076.121    1475 486.83     

## 5.5 — Export: Country KPIs

In [5]:
country_kpis = (
    df_sales.groupby('country').agg(
        revenue          = ('revenue',   'sum'),
        orders           = ('invoice',   'nunique'),
        unique_customers = ('customer_id','nunique'),
        items_sold       = ('quantity',  'sum')
    ).reset_index()
)
country_kpis['aov']            = (country_kpis['revenue'] / country_kpis['orders']).round(2)
country_kpis['revenue_share']  = (country_kpis['revenue'] / country_kpis['revenue'].sum() * 100).round(2)

# Cancellation rate by country
cancel_country = (
    df.groupby('country').agg(
        total_inv  = ('invoice','nunique'),
        cancel_inv = ('is_cancelled','sum')
    ).reset_index()
)
cancel_country['cancellation_rate'] = (cancel_country['cancel_inv'] / cancel_country['total_inv'] * 100).round(2)
country_kpis = country_kpis.merge(cancel_country[['country','cancellation_rate']], on='country', how='left')

country_kpis.sort_values('revenue', ascending=False).to_csv(PROCESSED_DIR / 'country_kpis.csv', index=False)
print('Saved: country_kpis.csv')
print(country_kpis.sort_values('revenue', ascending=False).head(10).to_string(index=False))

Saved: country_kpis.csv
       country      revenue  orders  unique_customers  items_sold     aov  revenue_share  cancellation_rate
United Kingdom 17410196.117   36535              5351     9187753  476.53          85.03              37.15
          Eire   658767.310     626                 6      336329 1052.34           3.22              63.03
   Netherlands   554038.090     228                22      383879 2429.99           2.71              18.88
       Germany   425019.711     789               107      225154  538.68           2.08              82.10
        France   350456.090     622                96      271901  563.43           1.71              51.61
     Australia   169283.460      95                15      103759 1781.93           0.83              83.76
         Spain   108332.490     154                41       50307  703.46           0.53              48.40
   Switzerland   100685.590      93                23       52762 1082.64           0.49              42.28
    

## 5.6 — Export: Product KPIs

In [6]:
product_kpis = (
    df_sales.groupby(['stockcode','description','category']).agg(
        revenue    = ('revenue',  'sum'),
        qty_sold   = ('quantity', 'sum'),
        orders     = ('invoice',  'nunique'),
        avg_price  = ('price',    'mean')
    ).reset_index()
)
# Cancellation rate per product
cancel_prod = (
    df.groupby('stockcode').agg(
        total_lines  = ('invoice', 'count'),
        cancel_lines = ('is_cancelled','sum')
    ).reset_index()
)
cancel_prod['cancel_rate'] = (cancel_prod['cancel_lines'] / cancel_prod['total_lines'] * 100).round(2)
product_kpis = product_kpis.merge(cancel_prod[['stockcode','cancel_rate']], on='stockcode', how='left')
product_kpis['revenue_share'] = (product_kpis['revenue'] / product_kpis['revenue'].sum() * 100).round(4)
product_kpis['avg_price']     = product_kpis['avg_price'].round(2)

product_kpis.sort_values('revenue', ascending=False).to_csv(PROCESSED_DIR / 'product_kpis.csv', index=False)
print('Saved: product_kpis.csv')
print(f'Total products: {len(product_kpis):,}')
product_kpis.sort_values('revenue', ascending=False).head(10).to_string(index=False)

Saved: product_kpis.csv
Total products: 5,600


"stockcode                        description category   revenue  qty_sold  orders  avg_price  cancel_rate  revenue_share\n        M                             Manual   Manual 339226.24      9629     784     394.32        38.70         1.6567\n    22423           Regency Cakestand 3 Tier  Regency 330590.32     26478    3918      14.19         7.95         1.6145\n      DOT                     Dotcom Postage   Dotcom 309854.11      1415    1415     218.98         0.21         1.5132\n   85123A White Hanging Heart T-Light Holder    White 257546.20     94142    5356       3.08         2.37         1.2578\n    23843        Paper Craft , Little Birdie    Paper 168469.60     80995       1       2.08        50.00         0.8228\n    47566                      Party Bunting    Party 148318.28     28200    2674       5.72         0.84         0.7243\n   85099B            Jumbo Bag Red Retrospot    Jumbo 145961.83     77280    3245       2.37         2.04         0.7128\n    84879      Assorted

## 5.7 — Export: RFM Segments

In [7]:
rfm_df = df_sales[df_sales['customer_id'] != 'GUEST'].copy()
snapshot_date = rfm_df['invoicedate'].max() + pd.Timedelta(days=1)

rfm = rfm_df.groupby('customer_id').agg(
    recency   = ('invoicedate', lambda x: (snapshot_date - x.max()).days),
    frequency = ('invoice',     'nunique'),
    monetary  = ('revenue',     'sum')
).reset_index()

rfm['r_score'] = pd.qcut(rfm['recency'],   q=4, labels=[4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'),  q=4, labels=[1,2,3,4]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

def rfm_segment(score):
    if score >= 10: return 'Champions'
    elif score >= 8: return 'Loyal Customers'
    elif score >= 6: return 'Potential Loyalists'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['segment'] = rfm['rfm_score'].apply(rfm_segment)
rfm['monetary'] = rfm['monetary'].round(2)

rfm.to_csv(PROCESSED_DIR / 'rfm_segments.csv', index=False)
print('Saved: rfm_segments.csv')
print(rfm['segment'].value_counts().to_string())

Saved: rfm_segments.csv
segment
Champions              1740
Potential Loyalists    1216
Loyal Customers        1186
At Risk                1165
Lost                    571


## 5.8 — Export: Main Tableau File

In [8]:
# Full transaction-level file — this is the primary data source in Tableau
# Tableau will use this for all calculated fields and filters
tableau_cols = [
    'invoice', 'stockcode', 'description', 'category',
    'quantity', 'invoicedate', 'price', 'revenue', 'abs_revenue',
    'customer_id', 'country',
    'year', 'month', 'month_name', 'quarter', 'day_of_week', 'hour',
    'is_cancelled'
]
# Only keep columns that exist in df
tableau_cols = [c for c in tableau_cols if c in df.columns]

tableau_df = df[tableau_cols].copy()
tableau_df.to_csv(PROCESSED_DIR / 'tableau_main.csv', index=False)

print('Saved: tableau_main.csv')
print(f'Shape: {tableau_df.shape}')
print(f'Columns: {list(tableau_df.columns)}')
tableau_df.head(5)

Saved: tableau_main.csv
Shape: (1027017, 18)
Columns: ['invoice', 'stockcode', 'description', 'category', 'quantity', 'invoicedate', 'price', 'revenue', 'abs_revenue', 'customer_id', 'country', 'year', 'month', 'month_name', 'quarter', 'day_of_week', 'hour', 'is_cancelled']


,invoice,stockcode,description,category,quantity,invoicedate,price,revenue,abs_revenue,customer_id,country,year,month,month_name,quarter,day_of_week,hour,is_cancelled
0,489434,85048,15Cm Christmas Glass Ball 20 Lights,15Cm,12,2009-12-01 07:45:00,6.95,83.4,83.4,13085,United Kingdom,2009,12,December,4,Tuesday,7,False
1,489434,79323P,Pink Cherry Lights,Pink,12,2009-12-01 07:45:00,6.75,81.0,81.0,13085,United Kingdom,2009,12,December,4,Tuesday,7,False
2,489434,79323W,White Cherry Lights,White,12,2009-12-01 07:45:00,6.75,81.0,81.0,13085,United Kingdom,2009,12,December,4,Tuesday,7,False
3,489434,22041,"Record Frame 7"" Single Size",Record,48,2009-12-01 07:45:00,2.10,100.8,100.8,13085,United Kingdom,2009,12,December,4,Tuesday,7,False
4,489434,21232,Strawberry Ceramic Trinket Box,Strawberry,24,2009-12-01 07:45:00,1.25,30.0,30.0,13085,United Kingdom,2009,12,December,4,Tuesday,7,False


## 5.9 — Final Export Summary

In [9]:
exports = [
    'kpi_summary.csv',
    'monthly_kpis.csv',
    'country_kpis.csv',
    'product_kpis.csv',
    'rfm_segments.csv',
    'tableau_main.csv',
    'cleaned_retail.csv'
]

print('=== DATA/PROCESSED EXPORT MANIFEST ===')
for fname in exports:
    fpath = PROCESSED_DIR / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        rows = pd.read_csv(fpath).shape[0]
        print(f'  {fname:<30} {rows:>8,} rows  {size_kb:>8.1f} KB  ✓')
    else:
        print(f'  {fname:<30} NOT FOUND — re-run this notebook')

print()
print('All files ready. Connect tableau_main.csv to Tableau Public as primary data source.')
print('Use country_kpis.csv and monthly_kpis.csv as supplementary sources for pre-aggregated views.')

=== DATA/PROCESSED EXPORT MANIFEST ===
  kpi_summary.csv                       1 rows       0.2 KB  ✓
  monthly_kpis.csv                     25 rows       1.6 KB  ✓
  country_kpis.csv                     43 rows       2.1 KB  ✓
  product_kpis.csv                  5,600 rows     404.2 KB  ✓
  rfm_segments.csv                  5,878 rows     227.0 KB  ✓


  tableau_main.csv               1,027,017 rows  141614.8 KB  ✓


  cleaned_retail.csv             1,027,017 rows  143578.6 KB  ✓

All files ready. Connect tableau_main.csv to Tableau Public as primary data source.
Use country_kpis.csv and monthly_kpis.csv as supplementary sources for pre-aggregated views.
